# LeetCode 2093 — Minimum Cost to Reach City With Discounts bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-to-reach-city-with-discounts/ · Độ khó: **Medium** · Chủ đề: Graph, Shortest Path, Heap (Priority Queue)

> **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def minimumCost(self, n: int, highways: List[List[int]], discounts: int) -> int:
        ...
```

Lộ trình:
1. **Brute force**: liệt kê mọi đường đi đơn từ `0` tới `n−1`, trên mỗi đường dùng phiếu giảm giá cho các cạnh đắt nhất.
2. **Bellman-Ford theo lớp**: `dist[k][v]` = chi phí nhỏ nhất tới `v` đã dùng `k` phiếu, nới lỏng lặp.
3. **Dijkstra trên đồ thị trạng thái** `(thành phố, số phiếu đã dùng)` — lời giải chuẩn.
4. **Dijkstra + cắt tỉa**: giới hạn số phiếu ở `n − 1` và bỏ trạng thái bị "trội" (đến cùng thành phố, tốn nhiều hơn mà còn ít phiếu hơn).

## 1. Mục tiêu học

1. Nhận ra khi đường đi ngắn nhất có **tài nguyên hữu hạn** (phiếu giảm giá), trạng thái phải gồm cả **lượng tài nguyên đã dùng**: `(đỉnh, k)`.
2. Dựng **đồ thị phân lớp** ngầm: mỗi cạnh `(u, v, w)` cho hai cách đi — giữ lớp (`+w`) hoặc lên lớp (`+w // 2`).
3. Cài Dijkstra với `heapq` trên trạng thái tuple, xử lý "bản ghi cũ" trong heap.
4. Hiểu vì sao "đường ngắn nhất theo giá gốc rồi giảm giá các cạnh đắt nhất" **sai**, và kiểm chứng bằng phản ví dụ thật.
5. Biết các tối ưu: giới hạn `k ≤ n − 1`, cắt tỉa trạng thái bị trội.

## 2. Đề bài & ví dụ

Có `n` thành phố `0..n−1` và các đường cao tốc **hai chiều** `highways[i] = [a, b, toll]`. Bạn có `discounts` phiếu giảm giá; dùng một phiếu trên một cạnh thì phí cạnh đó còn `toll // 2` (chia nguyên). Mỗi phiếu dùng một lần, mỗi lần đi qua cạnh dùng tối đa một phiếu. Trả về chi phí nhỏ nhất đi từ `0` tới `n − 1`, hoặc `-1` nếu không tới được.

**Ví dụ 1:** `n = 5`, `highways = [[0,1,4],[2,1,3],[1,4,11],[3,2,3],[3,4,2]]`, `discounts = 1` → `9` (`0→1` giá 4, `1→4` dùng phiếu `11 // 2 = 5`).

**Ví dụ 2:** `n = 4`, `highways = [[1,3,17],[1,2,7],[3,2,5],[0,1,6],[3,0,20]]`, `discounts = 20` → `8` (`3 + 3 + 2`).

**Ví dụ 3:** `n = 4`, `highways = [[0,1,3],[2,3,2]]`, `discounts = 0` → `-1`.

**Ràng buộc:** `2 ≤ n ≤ 1000`, `1 ≤ len(highways) ≤ 1000`, `0 ≤ toll ≤ 10⁵`, `0 ≤ discounts ≤ 500`, không có cạnh trùng.

In [1]:
import heapq
import random
import time
from typing import List

examples = [((5, [[0, 1, 4], [2, 1, 3], [1, 4, 11], [3, 2, 3], [3, 4, 2]], 1), 9),
            ((4, [[1, 3, 17], [1, 2, 7], [3, 2, 5], [0, 1, 6], [3, 0, 20]], 20), 8),
            ((4, [[0, 1, 3], [2, 3, 2]], 0), -1)]
for (n, hw, d), want in examples:
    print(f"n = {n}, discounts = {d}, highways = {hw} -> {want}")

n = 5, discounts = 1, highways = [[0, 1, 4], [2, 1, 3], [1, 4, 11], [3, 2, 3], [3, 4, 2]] -> 9
n = 4, discounts = 20, highways = [[1, 3, 17], [1, 2, 7], [3, 2, 5], [0, 1, 6], [3, 0, 20]] -> 8
n = 4, discounts = 0, highways = [[0, 1, 3], [2, 3, 2]] -> -1


## 3. Kiến thức nền

### 3.1 Danh sách kề cho đồ thị vô hướng có trọng số

In [2]:
def build_graph(n, highways):
    g = [[] for _ in range(n)]
    for a, b, w in highways:
        g[a].append((b, w))
        g[b].append((a, w))
    return g

g = build_graph(*examples[0][0][:2])
for v, nb_ in enumerate(g):
    print(v, nb_)

0 [(1, 4)]
1 [(0, 4), (2, 3), (4, 11)]
2 [(1, 3), (3, 3)]
3 [(2, 3), (4, 2)]
4 [(1, 11), (3, 2)]


### 3.2 Dijkstra cơ bản với `heapq`

Heap lưu `(khoảng_cách, đỉnh)`. Lấy ra đỉnh có khoảng cách nhỏ nhất; nếu khoảng cách lấy ra lớn hơn giá trị đã biết thì đó là **bản ghi cũ** — bỏ qua.

In [3]:
def dijkstra(g, src):
    INF = float("inf"); dist = [INF] * len(g); dist[src] = 0; pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w; heapq.heappush(pq, (d + w, v))
    return dist

print("không giảm giá, ví dụ 1:", dijkstra(g, 0))

không giảm giá, ví dụ 1: [0, 4, 7, 10, 12]


### 3.3 Heap với tuple nhiều thành phần

`heapq` so sánh tuple theo thứ tự từ điển — `(chi_phí, đỉnh, k)` được sắp theo chi phí trước. Mọi phần tử phải so sánh được (số nguyên là ổn).

In [4]:
h = []
for item in [(9, 4, 1), (4, 1, 0), (5, 1, 1), (4, 0, 0)]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(len(h))])

[(4, 0, 0), (4, 1, 0), (5, 1, 1), (9, 4, 1)]


### 3.4 Chia nguyên `//`

Phí sau giảm là `toll // 2` (làm tròn xuống): `11 // 2 = 5`, `5 // 2 = 2`, `1 // 2 = 0`.

## 4. Ý tưởng nền tảng / trực giác

Trên **một đường đi cố định** với `L` cạnh, nên dùng phiếu cho `min(D, L)` cạnh có **mức tiết kiệm** `w − w // 2` lớn nhất — tức các cạnh đắt nhất. Nhưng đường tối ưu **phụ thuộc** vào số phiếu: có phiếu thì một đường có cạnh rất đắt có thể trở nên rẻ.

Ví dụ 1 với các mức phiếu khác nhau:

| `discounts` | đường tốt nhất | chi phí |
|---|---|---|
| 0 | `0 → 1 → 2 → 3 → 4` (`4 + 3 + 3 + 2`) | 12 |
| 1 | `0 → 1 → 4` (`4 + 11//2`) | 9 |
| 2 | `0 → 1 → 4` (`4//2 + 11//2`) | 7 |

Với `D = 0`, đường dài 4 cạnh rẻ hơn (`12 < 15`); với `D = 1`, đường 2 cạnh thắng. ⇒ Không thể chọn đường trước rồi mới giảm giá.

In [5]:
def path_cost(path_edges, D):
    """Chi phí tốt nhất trên một đường cố định: giảm giá các cạnh tiết kiệm nhiều nhất."""
    save = sorted((w - w // 2 for w in path_edges), reverse=True)
    return sum(path_edges) - sum(save[:D])

for D in (0, 1, 2):
    print(f"D = {D}: đường 0-1-4 tốn {path_cost([4, 11], D)}, đường 0-1-2-3-4 tốn {path_cost([4, 3, 3, 2], D)}")

D = 0: đường 0-1-4 tốn 15, đường 0-1-2-3-4 tốn 12
D = 1: đường 0-1-4 tốn 9, đường 0-1-2-3-4 tốn 10
D = 2: đường 0-1-4 tốn 7, đường 0-1-2-3-4 tốn 8


## 5. Cách 1 — Brute Force / baseline: mọi đường đi đơn

**Ý tưởng:** DFS liệt kê mọi đường đi **đơn** (không lặp đỉnh) từ `0` tới `n − 1`; với mỗi đường tính `path_cost(cạnh, D)`; lấy nhỏ nhất. Đường tối ưu luôn có thể chọn là đường đơn: nếu đi lặp đỉnh, cắt bỏ chu trình không làm tăng chi phí (phí `≥ 0`) và không cần thêm phiếu.

In [6]:
def min_cost_brute(n, highways, discounts):
    g = build_graph(n, highways)
    best = float("inf")
    seen = [False] * n
    def dfs(u, edges):
        nonlocal best
        if u == n - 1:
            best = min(best, path_cost(edges, discounts)); return
        for v, w in g[u]:
            if not seen[v]:
                seen[v] = True; edges.append(w)
                dfs(v, edges)
                edges.pop(); seen[v] = False
    seen[0] = True
    dfs(0, [])
    return -1 if best == float("inf") else best

for args, want in examples:
    print(min_cost_brute(*args), "kỳ vọng", want)

9 kỳ vọng 9
8 kỳ vọng 8
-1 kỳ vọng -1


**Độ phức tạp:** số đường đi đơn có thể **mũ** theo `n` — chỉ dùng cho đồ thị rất nhỏ (≤ 9 đỉnh). Mỗi đường tốn `O(L log L)` để sắp mức tiết kiệm.

**Nút thắt:** liệt kê đường. Ta cần một cách "nhớ" đủ thông tin để không phải liệt kê: đó là **(đỉnh hiện tại, số phiếu đã dùng)**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** định nghĩa `dist[v][k]` = chi phí nhỏ nhất đi từ `0` tới `v` khi đã dùng **đúng** `k` phiếu. Từ trạng thái `(u, k)`, cạnh `(u, v, w)` cho hai chuyển trạng thái:

- `(u, k) → (v, k)` với chi phí `w` (không dùng phiếu);
- `(u, k) → (v, k + 1)` với chi phí `w // 2` nếu `k < D`.

Đó là **đồ thị phân lớp**: `D + 1` bản sao của đồ thị gốc, cạnh giảm giá nối lớp `k` lên lớp `k + 1`. Mọi trọng số `≥ 0` ⇒ chạy **Dijkstra** trên đồ thị này. Đáp án `= min_k dist[n−1][k]`.

**Quy tắc:**
- *Nếu* lấy ra trạng thái `(n−1, k)` lần đầu *thì* đó là đáp án (Dijkstra lấy ra theo chi phí tăng dần).
- *Nếu* `D ≥ n − 1` *thì* coi như `D = n − 1` (đường đơn có tối đa `n − 1` cạnh, phiếu thừa vô dụng).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bellman-Ford theo lớp

Nới lỏng mọi cạnh, mọi lớp, lặp tới khi không đổi (tối đa `n·(D+1)` vòng). Dễ hiểu, không cần heap; chậm hơn Dijkstra nhiều.

In [7]:
def min_cost_bellman(n, highways, discounts):
    D = min(discounts, n - 1)
    INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]
    dist[0][0] = 0
    edges = [(a, b, w) for a, b, w in highways] + [(b, a, w) for a, b, w in highways]
    for _ in range(n * (D + 1)):
        changed = False
        for u, v, w in edges:
            du = dist[u]; dv = dist[v]
            for k in range(D + 1):
                if du[k] == INF:
                    continue
                if du[k] + w < dv[k]:
                    dv[k] = du[k] + w; changed = True
                if k < D and du[k] + w // 2 < dv[k + 1]:
                    dv[k + 1] = du[k] + w // 2; changed = True
        if not changed:
            break
    best = min(dist[n - 1])
    return -1 if best == INF else best

for args, want in examples:
    print(min_cost_bellman(*args), "kỳ vọng", want)

9 kỳ vọng 9
8 kỳ vọng 8
-1 kỳ vọng -1


**Độ phức tạp:** mỗi vòng `O(E · D)`; số vòng tối đa bằng số cạnh trên đường tối ưu trong đồ thị trạng thái — tệ nhất `O(n D)` vòng ⇒ `O(n · E · D²)`. Thực tế dừng sớm, nhưng vẫn chậm hơn Dijkstra.

### 7.2 Cách 3 — Dijkstra trên trạng thái `(v, k)` (code nộp)

In [8]:
def min_cost_dijkstra(n, highways, discounts):
    g = build_graph(n, highways)
    D = min(discounts, n - 1)
    INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]
    dist[0][0] = 0
    pq = [(0, 0, 0)]                           # (chi phí, thành phố, số phiếu đã dùng)
    while pq:
        d, u, k = heapq.heappop(pq)
        if d > dist[u][k]:
            continue
        if u == n - 1:
            return d                           # lần đầu lấy ra đích = tối ưu
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v][k]:
                dist[v][k] = nd; heapq.heappush(pq, (nd, v, k))
            if k < D:
                nd = d + w // 2
                if nd < dist[v][k + 1]:
                    dist[v][k + 1] = nd; heapq.heappush(pq, (nd, v, k + 1))
    return -1

for args, want in examples:
    print(min_cost_dijkstra(*args), "kỳ vọng", want)

9 kỳ vọng 9
8 kỳ vọng 8
-1 kỳ vọng -1


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** số trạng thái `V' = n (D + 1)`, số cạnh trạng thái `E' ≈ 4 E (D + 1)` ⇒ `O(E' log V')` ≈ `4·1000·501·log(5·10⁵) ≈ 4·10⁷` — chạy được (thực tế nhanh hơn nhiều nhờ dừng sớm). Space `O(n D)`.

### 7.3 Cách 4 — Dijkstra + cắt tỉa trạng thái bị trội

Trạng thái `(v, k, d)` bị **trội** bởi `(v, k', d')` nếu `k' ≤ k` và `d' ≤ d` (tới cùng nơi, dùng ít phiếu hơn, tốn ít hơn). Khi lấy ra `(v, k)`, nếu đã từng lấy ra `v` với số phiếu `≤ k` thì trạng thái này vô ích. Vì Dijkstra lấy theo chi phí tăng dần, chỉ cần nhớ `minK[v]` = số phiếu nhỏ nhất từng được lấy ra tại `v`.

In [9]:
def min_cost_dominance(n, highways, discounts):
    g = build_graph(n, highways)
    D = min(discounts, n - 1)
    minK = [D + 1] * n                          # số phiếu nhỏ nhất đã "chốt" tại mỗi đỉnh
    pq = [(0, 0, 0)]
    pops = 0
    while pq:
        d, u, k = heapq.heappop(pq)
        if k >= minK[u]:
            continue                            # bị trội: đã tới u rẻ hơn (hoặc bằng) với ít phiếu hơn
        minK[u] = k; pops += 1
        if u == n - 1:
            return d
        for v, w in g[u]:
            if k < minK[v]:
                heapq.heappush(pq, (d + w, v, k))
            if k + 1 < minK[v] and k < D:
                heapq.heappush(pq, (d + w // 2, v, k + 1))
    return -1

for args, want in examples:
    print(min_cost_dominance(*args), "kỳ vọng", want)

9 kỳ vọng 9
8 kỳ vọng 8
-1 kỳ vọng -1


**Vì sao đúng?** Khi `(u, k)` được lấy ra với chi phí `d`, mọi trạng thái `(u, k₂)` lấy ra sau có chi phí `≥ d`. Nếu `k₂ ≥ k`, nó vừa đắt hơn vừa còn ít phiếu hơn ⇒ mọi đường đi tiếp từ nó đều có đường tương ứng từ `(u, k)` không tệ hơn. Mỗi đỉnh được lấy ra tối đa `D + 1` lần nhưng thường ít hơn nhiều.

**Độ phức tạp:** tệ nhất như Cách 3, thực tế ít trạng thái hơn.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1: in mỗi trạng thái lấy ra khỏi heap và các trạng thái được đẩy vào.

In [10]:
def trace_dijkstra(n, highways, discounts):
    g = build_graph(n, highways); D = min(discounts, n - 1); INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]; dist[0][0] = 0; pq = [(0, 0, 0)]
    while pq:
        d, u, k = heapq.heappop(pq)
        if d > dist[u][k]:
            print(f"  bỏ bản ghi cũ ({u}, k={k}) d={d}"); continue
        print(f"lấy ra: thành phố {u}, đã dùng {k} phiếu, chi phí {d}")
        if u == n - 1:
            print("=> tới đích:", d); return d
        for v, w in g[u]:
            pushed = []
            if d + w < dist[v][k]:
                dist[v][k] = d + w; heapq.heappush(pq, (d + w, v, k)); pushed.append(f"({v}, k={k}) = {d + w}")
            if k < D and d + w // 2 < dist[v][k + 1]:
                dist[v][k + 1] = d + w // 2; heapq.heappush(pq, (d + w // 2, v, k + 1)); pushed.append(f"({v}, k={k + 1}) = {d + w // 2}")
            if pushed:
                print(f"    cạnh {u}-{v} (toll {w}): đẩy {', '.join(pushed)}")
    return -1

trace_dijkstra(*examples[0][0])

lấy ra: thành phố 0, đã dùng 0 phiếu, chi phí 0
    cạnh 0-1 (toll 4): đẩy (1, k=0) = 4, (1, k=1) = 2
lấy ra: thành phố 1, đã dùng 1 phiếu, chi phí 2
    cạnh 1-0 (toll 4): đẩy (0, k=1) = 6
    cạnh 1-2 (toll 3): đẩy (2, k=1) = 5
    cạnh 1-4 (toll 11): đẩy (4, k=1) = 13
lấy ra: thành phố 1, đã dùng 0 phiếu, chi phí 4
    cạnh 1-2 (toll 3): đẩy (2, k=0) = 7
    cạnh 1-4 (toll 11): đẩy (4, k=0) = 15, (4, k=1) = 9
lấy ra: thành phố 2, đã dùng 1 phiếu, chi phí 5
    cạnh 2-3 (toll 3): đẩy (3, k=1) = 8
lấy ra: thành phố 0, đã dùng 1 phiếu, chi phí 6
lấy ra: thành phố 2, đã dùng 0 phiếu, chi phí 7
    cạnh 2-3 (toll 3): đẩy (3, k=0) = 10
lấy ra: thành phố 3, đã dùng 1 phiếu, chi phí 8
lấy ra: thành phố 4, đã dùng 1 phiếu, chi phí 9
=> tới đích: 9


9

Đọc trace: trạng thái `(4, k=1)` với chi phí `9` được tạo từ `(1, k=0)` qua cạnh `1-4` dùng phiếu (`4 + 5`). Trong khi đó `(4, k=0)` qua đường dài chỉ đạt `12`. Dijkstra lấy ra `(4, 1)` trước vì `9 < 12`.

Bảng `dist` cuối cùng (ví dụ 1, `D = 1`):

In [11]:
def dist_table(n, highways, discounts):
    g = build_graph(n, highways); D = min(discounts, n - 1); INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]; dist[0][0] = 0; pq = [(0, 0, 0)]
    while pq:
        d, u, k = heapq.heappop(pq)
        if d > dist[u][k]: continue
        for v, w in g[u]:
            if d + w < dist[v][k]: dist[v][k] = d + w; heapq.heappush(pq, (d + w, v, k))
            if k < D and d + w // 2 < dist[v][k + 1]: dist[v][k + 1] = d + w // 2; heapq.heappush(pq, (d + w // 2, v, k + 1))
    print("thành phố | " + " | ".join(f"k={k}" for k in range(D + 1)))
    for v in range(n):
        print(f"{v:>9} | " + " | ".join(f"{dist[v][k]:>3}" for k in range(D + 1)))

dist_table(*examples[0][0])

thành phố | k=0 | k=1
        0 |   0 |   6
        1 |   4 |   2
        2 |   7 |   5
        3 |  10 |   8
        4 |  12 |   9


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minimumCost(self, n: int, highways: List[List[int]], discounts: int) -> int:
        g = [[] for _ in range(n)]
        for a, b, w in highways:
            g[a].append((b, w))
            g[b].append((a, w))
        D = min(discounts, n - 1)               # phiếu thừa không có ích
        INF = float("inf")
        dist = [[INF] * (D + 1) for _ in range(n)]
        dist[0][0] = 0
        pq = [(0, 0, 0)]
        while pq:
            d, u, k = heapq.heappop(pq)
            if d > dist[u][k]:
                continue
            if u == n - 1:
                return d
            for v, w in g[u]:
                nd = d + w
                if nd < dist[v][k]:
                    dist[v][k] = nd
                    heapq.heappush(pq, (nd, v, k))
                if k < D:
                    nd = d + w // 2
                    if nd < dist[v][k + 1]:
                        dist[v][k + 1] = nd
                        heapq.heappush(pq, (nd, v, k + 1))
        return -1

sol = Solution()
for args, want in examples:
    got = sol.minimumCost(*args)
    print(got, "OK" if got == want else "SAI")

9 OK
8 OK
-1 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (tương ứng đường đi).** Mỗi hành trình hợp lệ trong đồ thị gốc (đi qua các cạnh, mỗi lần đi qua cạnh dùng tối đa một phiếu, tổng phiếu `≤ D`) tương ứng với một đường trong đồ thị trạng thái từ `(0, 0)` tới `(n−1, k)` với `k` = số phiếu đã dùng, **cùng chi phí** — và ngược lại. (Đi cạnh không phiếu ⇔ ở lại lớp; dùng phiếu ⇔ lên một lớp.)

**Bổ đề 2 (Dijkstra).** Đồ thị trạng thái có trọng số không âm (`w ≥ 0`, `w // 2 ≥ 0`) ⇒ Dijkstra lấy các trạng thái ra theo chi phí không giảm, và chi phí lúc lấy ra là chi phí ngắn nhất. Vì vậy trạng thái đích đầu tiên được lấy ra có chi phí `= min_k dist[n−1][k]`.

**Bổ đề 3 (giới hạn `D ≤ n − 1`).** Tồn tại hành trình tối ưu là đường đơn (cắt chu trình không làm tăng chi phí vì phí không âm và không cần thêm phiếu) — có tối đa `n − 1` cạnh, nên dùng tối đa `n − 1` phiếu. Giảm `D` về `min(D, n−1)` không đổi đáp án. ✓

**Cách 2 (Bellman-Ford):** nới lỏng lặp tới điểm bất động cho đúng khoảng cách ngắn nhất trên đồ thị trạng thái (không có chu trình âm). ✓

## 11. Phân tích Time / Space

`n ≤ 1000`, `E ≤ 1000`, `D ≤ 500` (sau khi cắt: `D' = min(D, n − 1)`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi đường đơn | mũ | `O(n)` đệ quy |
| 2. Bellman-Ford lớp | `O(vòng · E · D')` | `O(n D')` |
| 3. Dijkstra trạng thái | `O(E D' log(n D'))` | `O(n D')` |
| 4. Dijkstra + trội | `≤` Cách 3, thường nhanh hơn | `O(n + heap)` |

Đo trên đồ thị ngẫu nhiên `n = 1000`, `E = 1000` (một đường "xương sống" `0 → … → n−1` để đảm bảo liên thông) và `D = 500`:

In [13]:
def random_graph(n, E, rng, maxw=10**5):
    edges = set(); hw = []
    perm = list(range(1, n - 1)); rng.shuffle(perm)
    chain = [0] + perm[: min(len(perm), E // 2)] + [n - 1]
    for a, b in zip(chain, chain[1:]):
        edges.add((min(a, b), max(a, b))); hw.append([a, b, rng.randint(0, maxw)])
    while len(hw) < E:
        a, b = rng.sample(range(n), 2)
        if (min(a, b), max(a, b)) not in edges:
            edges.add((min(a, b), max(a, b))); hw.append([a, b, rng.randint(0, maxw)])
    return hw

rng = random.Random(1)
hw_big = random_graph(1000, 1000, rng)
for D in (0, 5, 500):
    res = []
    for name, fn in [("dijkstra", min_cost_dijkstra), ("dominance", min_cost_dominance), ("submit", Solution().minimumCost)]:
        t0 = time.perf_counter(); r = fn(1000, hw_big, D); res.append(r)
        print(f"D = {D:>3} {name:9s}: {r:>8}  {time.perf_counter() - t0:.2f}s")
    assert len(set(res)) == 1
t0 = time.perf_counter(); rb = min_cost_bellman(60, random_graph(60, 120, random.Random(2)), 10)
print(f"bellman (n = 60, E = 120, D = 10): {rb}, {time.perf_counter() - t0:.2f}s")

D =   0 dijkstra :   371530  0.00s
D =   0 dominance:   371530  0.00s
D =   0 submit   :   371530  0.00s
D =   5 dijkstra :   260041  0.00s
D =   5 dominance:   260041  0.00s
D =   5 submit   :   260041  0.00s
D = 500 dijkstra :   185761  0.05s
D = 500 dominance:   185761  0.00s
D = 500 submit   :   185761  0.05s
bellman (n = 60, E = 120, D = 10): 18771, 0.00s


## 12. Bộ test

In [14]:
fns = [min_cost_bellman, min_cost_dijkstra, min_cost_dominance, Solution().minimumCost]

for args, want in examples:
    assert min_cost_brute(*args) == want
    for fn in fns:
        assert fn(*args) == want

edge = [
    ((2, [[0, 1, 10]], 0), 10),
    ((2, [[0, 1, 10]], 5), 5),                     # phiếu thừa
    ((2, [[0, 1, 1]], 1), 0),                      # 1 // 2 = 0
    ((3, [[0, 1, 0], [1, 2, 0]], 0), 0),           # phí 0
    ((3, [[1, 2, 4]], 3), -1),                     # 0 cô lập
    ((3, [[0, 2, 100], [0, 1, 30], [1, 2, 30]], 0), 60),
    ((3, [[0, 2, 100], [0, 1, 30], [1, 2, 30]], 1), 45),   # 30//2 + 30 = 45 < 100//2 = 50
    ((3, [[0, 2, 100], [0, 1, 30], [1, 2, 30]], 2), 30),
]
for args, want in edge:
    assert min_cost_brute(*args) == want, args
    for fn in fns:
        assert fn(*args) == want, (fn, args)

rng = random.Random(2024)
for _ in range(400):
    n = rng.randint(2, 7)
    pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    hw = [[a, b, rng.randint(0, 30)] for a, b in rng.sample(pairs, rng.randint(1, len(pairs)))]
    for e in hw:
        if rng.random() < 0.5: e[0], e[1] = e[1], e[0]
    D = rng.randint(0, 4)
    want = min_cost_brute(n, hw, D)
    for fn in fns:
        assert fn(n, hw, D) == want, (fn, n, hw, D)
print("ví dụ + edge + 400 đồ thị ngẫu nhiên khớp brute force (liệt kê đường đơn)")

ví dụ + edge + 400 đồ thị ngẫu nhiên khớp brute force (liệt kê đường đơn)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tìm đường ngắn nhất theo giá gốc rồi mới giảm giá các cạnh đắt nhất

In [15]:
def wrong_path_then_discount(n, highways, discounts):
    g = build_graph(n, highways); INF = float("inf")
    dist = [INF] * n; par = [-1] * n; pw = [0] * n; dist[0] = 0; pq = [(0, 0)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w; par[v] = u; pw[v] = w; heapq.heappush(pq, (d + w, v))
    if dist[n - 1] == INF: return -1
    edges, v = [], n - 1
    while v != 0:
        edges.append(pw[v]); v = par[v]
    return path_cost(edges, discounts)                         # SAI: đường tối ưu phụ thuộc số phiếu

for args, want in examples:
    print("sai =", wrong_path_then_discount(*args), "| đúng =", want)

sai = 10 | đúng = 9
sai = 8 | đúng = 8
sai = -1 | đúng = -1


Ví dụ 1: đường rẻ nhất theo giá gốc là `0-1-2-3-4` (`12`); giảm cạnh đắt nhất (`4`) còn `10`, nhưng đường `0-1-4` với phiếu chỉ `9`. **Sửa:** để số phiếu là một phần của trạng thái.

### Lỗi 2 — Đánh dấu "đã thăm" theo **thành phố** (bỏ chiều số phiếu)

In [16]:
def wrong_visited_by_city(n, highways, discounts):
    g = build_graph(n, highways); done = [False] * n; pq = [(0, 0, 0)]
    while pq:
        d, u, k = heapq.heappop(pq)
        if done[u]: continue
        done[u] = True                                         # SAI: (u, k) khác (u, k')
        if u == n - 1: return d
        for v, w in g[u]:
            if not done[v]:
                heapq.heappush(pq, (d + w, v, k))
                if k < discounts: heapq.heappush(pq, (d + w // 2, v, k + 1))
    return -1

rng = random.Random(7); found = None
for _ in range(5000):
    n = rng.randint(3, 6); pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    hw = [[a, b, rng.randint(0, 30)] for a, b in rng.sample(pairs, rng.randint(2, len(pairs)))]
    D = rng.randint(1, 2)
    if wrong_visited_by_city(n, hw, D) != min_cost_brute(n, hw, D):
        found = (n, hw, D); break
print("phản ví dụ:", found, "| sai =", wrong_visited_by_city(*found), "| đúng =", min_cost_brute(*found))

phản ví dụ: (5, [[1, 4, 17], [0, 1, 3], [0, 2, 11], [3, 4, 18]], 1) | sai = 18 | đúng = 11


Đến thành phố `u` sớm nhất (rẻ nhất) có thể đã **tiêu hết phiếu**; một đường đắt hơn chút tới `u` nhưng còn phiếu có thể thắng ở phần sau. Đánh dấu theo `u` sẽ chặn đường đó. **Sửa:** `dist[u][k]` theo cặp, hoặc dùng quy tắc trội đúng (Cách 4: chỉ chặn khi số phiếu đã dùng **không ít hơn**).

### Lỗi 3 — Làm tròn **lên** khi giảm giá (`(w + 1) // 2` hoặc `ceil(w / 2)`)

In [17]:
def wrong_ceil(n, highways, discounts):
    hw = [[a, b, w] for a, b, w in highways]
    g = build_graph(n, hw); D = min(discounts, n - 1); INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]; dist[0][0] = 0; pq = [(0, 0, 0)]
    while pq:
        d, u, k = heapq.heappop(pq)
        if d > dist[u][k]: continue
        if u == n - 1: return d
        for v, w in g[u]:
            if d + w < dist[v][k]: dist[v][k] = d + w; heapq.heappush(pq, (d + w, v, k))
            half = (w + 1) // 2                                # SAI: đề dùng chia nguyên (làm tròn xuống)
            if k < D and d + half < dist[v][k + 1]: dist[v][k + 1] = d + half; heapq.heappush(pq, (d + half, v, k + 1))
    return -1

for args, want in examples:
    print("sai =", wrong_ceil(*args), "| đúng =", want)

sai = 10 | đúng = 9
sai = 10 | đúng = 8
sai = -1 | đúng = -1


`11 → 6` thay vì `5`. **Sửa:** `w // 2`.

### Lỗi 4 — Chỉ thêm cạnh một chiều

In [18]:
def wrong_directed(n, highways, discounts):
    g = [[] for _ in range(n)]
    for a, b, w in highways:
        g[a].append((b, w))                                    # SAI: đường cao tốc hai chiều
    D = min(discounts, n - 1); INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]; dist[0][0] = 0; pq = [(0, 0, 0)]
    while pq:
        d, u, k = heapq.heappop(pq)
        if d > dist[u][k]: continue
        if u == n - 1: return d
        for v, w in g[u]:
            if d + w < dist[v][k]: dist[v][k] = d + w; heapq.heappush(pq, (d + w, v, k))
            if k < D and d + w // 2 < dist[v][k + 1]: dist[v][k + 1] = d + w // 2; heapq.heappush(pq, (d + w // 2, v, k + 1))
    return -1

for args, want in examples:
    print("sai =", wrong_directed(*args), "| đúng =", want)

sai = 9 | đúng = 9
sai = 11 | đúng = 8
sai = -1 | đúng = -1


Ví dụ 1 tình cờ vẫn đúng vì đường tối ưu `0 → 1 → 4` đi đúng chiều các cạnh đã cho. Ví dụ 2 thì hỏng: đường tối ưu `0 → 1 → 2 → 3` cần đi cạnh `[3, 2, 5]` theo chiều `2 → 3`, ngược chiều khai báo ⇒ chỉ còn đường đắt hơn. **Sửa:** thêm cả hai chiều.

### Lỗi 5 — Lấy `dist[n−1][D]` (bắt buộc dùng hết phiếu) thay vì `min_k`

Nếu cài Bellman-Ford/DP rồi trả `dist[n−1][D]`, khi đường tối ưu có **ít cạnh hơn** `D` thì trạng thái "dùng đúng `D` phiếu" không tồn tại ⇒ trả `-1` hoặc giá trị sai.

In [19]:
def wrong_exact_D(n, highways, discounts):
    D = discounts; INF = float("inf")
    dist = [[INF] * (D + 1) for _ in range(n)]; dist[0][0] = 0
    edges = [(a, b, w) for a, b, w in highways] + [(b, a, w) for a, b, w in highways]
    for _ in range(n * (D + 1)):
        for u, v, w in edges:
            for k in range(D + 1):
                if dist[u][k] + w < dist[v][k]: dist[v][k] = dist[u][k] + w
                if k < D and dist[u][k] + w // 2 < dist[v][k + 1]: dist[v][k + 1] = dist[u][k] + w // 2
    return -1 if dist[n - 1][D] == INF else dist[n - 1][D]    # SAI: phải min theo k

for args, want in [examples[0], ((2, [[0, 1, 10]], 3), 5), ((3, [[0, 2, 100], [0, 1, 30], [1, 2, 30]], 0), 60)]:
    print("sai =", wrong_exact_D(*args), "| đúng =", want)

sai = 9 | đúng = 9
sai = 15 | đúng = 5
sai = 60 | đúng = 60


**Sửa:** `min(dist[n−1][k] for k in range(D + 1))` (Dijkstra dừng ở trạng thái đích đầu tiên đã tự làm điều này).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường ngắn nhất + được dùng tối đa `K` lần một quyền đặc biệt" ⇒ **Dijkstra trên `(đỉnh, số lần đã dùng)`** (đồ thị phân lớp).
- Cắt `K` về `n − 1` (hoặc số cạnh tối đa có ích).
- Quy tắc trội: cùng đỉnh, ít tài nguyên đã dùng hơn và rẻ hơn ⇒ trạng thái kia vô dụng.
- Đáp án là `min` theo mọi lớp, trừ khi đề bắt buộc dùng đúng `K`.
- Họ bài: 787 *Cheapest Flights Within K Stops* (giới hạn số cạnh), 1293 *Shortest Path in a Grid with Obstacles Elimination*, 1928 *Minimum Cost to Reach Destination in Time*, 2714 *Find Shortest Path with K Hops*.

## 15. Biến thể của bài

1. **Phiếu giảm theo tỉ lệ khác** (ví dụ còn `w // 3`): chỉ đổi công thức chuyển lớp.
2. **Hai loại phiếu** (giảm nửa `A` phiếu và miễn phí `B` phiếu): trạng thái `(v, a, b)` (Bài 5).
3. **Đồ thị có hướng:** bỏ chiều ngược.
4. **787 Cheapest Flights Within K Stops:** giới hạn số **cạnh** thay vì số phiếu — lớp là số cạnh đã đi.
5. **Bắt buộc dùng đúng `K` phiếu** (phiếu "phải tiêu"): trả `dist[n−1][K]` — đúng là "Lỗi 5" trở thành yêu cầu.

In [20]:
def cheapest_flights_k_stops(n, flights, src, dst, K):
    """Biến thể 4 (LeetCode 787): tối đa K điểm dừng = tối đa K+1 chuyến bay; Bellman-Ford K+1 vòng."""
    INF = float("inf"); dist = [INF] * n; dist[src] = 0
    for _ in range(K + 1):
        nd = dist[:]
        for a, b, w in flights:
            if dist[a] + w < nd[b]: nd[b] = dist[a] + w
        dist = nd
    return -1 if dist[dst] == INF else dist[dst]

print("787 ví dụ:", cheapest_flights_k_stops(4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1), "(kỳ vọng 700)")

787 ví dụ: 700 (kỳ vọng 700)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đường đơn | định nghĩa | mũ | `O(n)` | kiểm chứng |
| 2. Bellman-Ford lớp | nới lỏng lặp | `O(vòng · E D)` | `O(n D)` | dễ cài, đồ thị nhỏ |
| 3. Dijkstra `(v, k)` | đồ thị phân lớp | `O(E D log(n D))` | `O(n D)` | **nộp bài** |
| 4. Dijkstra + trội | cắt trạng thái | `≤` Cách 3 | `O(n + heap)` | `D` lớn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** ví dụ 1 với `discounts = 0`, `2`, `3`?

**Bài 2 (tính tay):** `n = 4`, `highways = [[0,1,8],[1,3,8],[0,2,1],[2,3,20]]`. Tính đáp án với `discounts = 0, 1, 2` và chỉ ra đường.

**Bài 3 (điền TODO):** hoàn thiện `best_on_fixed_path(tolls, D)` = chi phí nhỏ nhất trên một đường cố định với danh sách phí `tolls` và `D` phiếu.

**Bài 4 (tự viết lại):** viết `min_cost_with_path(n, highways, discounts)` trả về `(chi phí, danh sách thành phố trên đường, danh sách cạnh dùng phiếu)`; kiểm tra chi phí tính lại từ đường khớp.

**Bài 5 (mini challenge):** có `A` phiếu **giảm nửa** và `B` phiếu **miễn phí** (phí `0`); mỗi lần đi qua cạnh dùng tối đa một phiếu (một loại). Tìm chi phí nhỏ nhất bằng Dijkstra trên `(v, a, b)`; kiểm chứng bằng brute force liệt kê đường đơn. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [21]:
def best_on_fixed_path(tolls, D):
    # TODO: sắp mức tiết kiệm (w - w // 2) giảm dần, trừ D giá trị lớn nhất khỏi tổng
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Dùng bảng ở mục 4. (2) `D = 0` ⇒ `12`; `D = 2` ⇒ `2 + 5 = 7`. (3) `D = 3`: đường `0-1-4` chỉ có 2 cạnh (tốt nhất `7`); đường 4 cạnh phí `4, 3, 3, 2` giảm ba cạnh đầu ⇒ `2 + 1 + 1 + 2 = 6` ⇒ đáp án `6` — phiếu nhiều lại làm đường dài thắng.

**Bài 2:** (1) Hai đường: `0-1-3` (`8, 8`) và `0-2-3` (`1, 20`). (2) `D = 0`: `16` vs `21`; `D = 1`: `4 + 8 = 12` vs `1 + 10 = 11`. (3) `D = 2`: `4 + 4 = 8` vs `0 + 10 = 10`.

**Bài 3:** (1) `save = sorted((w - w//2 for w in tolls), reverse=True)`. (2) `sum(tolls) - sum(save[:D])`. (3) Đúng vì các cạnh độc lập nhau trên đường cố định.

**Bài 4:** (1) Lưu `parent[(v, k)] = (u, k_trước, có_dùng_phiếu)`. (2) Truy ngược từ trạng thái đích đầu tiên lấy ra. (3) Tính lại chi phí theo đường và tập cạnh giảm giá.

**Bài 5:** (1) `dist[v][a][b]`. (2) Ba chuyển: `+w`, `+w//2` (`a+1`), `+0` (`b+1`). (3) Brute force: trên mỗi đường đơn, thử mọi cách gán phiếu — hoặc tham lam: miễn phí cho `B` cạnh đắt nhất, rồi giảm nửa cho `A` cạnh đắt nhất còn lại (đúng vì mức tiết kiệm `w` của miễn phí luôn `≥` mức tiết kiệm `w − w//2` của giảm nửa).

## 19. Đáp án tham khảo

In [22]:
ex1 = examples[0][0][:2]
# Bài 1
assert [Solution().minimumCost(*ex1, D) for D in (0, 1, 2, 3)] == [12, 9, 7, 6]
assert [min_cost_brute(*ex1, D) for D in (0, 1, 2, 3)] == [12, 9, 7, 6]
# Bài 2
hw2 = [[0, 1, 8], [1, 3, 8], [0, 2, 1], [2, 3, 20]]
assert [Solution().minimumCost(4, hw2, D) for D in (0, 1, 2)] == [16, 11, 8]

# Bài 3
def best_on_fixed_path_ans(tolls, D):
    save = sorted((w - w // 2 for w in tolls), reverse=True)
    return sum(tolls) - sum(save[:D])
from itertools import combinations
rng = random.Random(31)
for _ in range(500):
    tolls = [rng.randint(0, 40) for _ in range(rng.randint(1, 7))]; D = rng.randint(0, 8)
    brute = min(sum(tolls) - sum(tolls[i] - tolls[i] // 2 for i in c)
                for r in range(min(D, len(tolls)) + 1) for c in combinations(range(len(tolls)), r))
    assert best_on_fixed_path_ans(tolls, D) == brute

# Bài 4
def min_cost_with_path_ans(n, highways, discounts):
    g = build_graph(n, highways); D = min(discounts, n - 1); INF = float("inf")
    dist = {(0, 0): 0}; par = {}; pq = [(0, 0, 0)]
    while pq:
        d, u, k = heapq.heappop(pq)
        if d > dist[(u, k)]: continue
        if u == n - 1:
            path, used, st = [u], [], (u, k)
            while st in par:
                pu, pk, disc, w = par[st]
                if disc: used.append((pu, st[0], w))
                path.append(pu); st = (pu, pk)
            return d, path[::-1], used[::-1]
        for v, w in g[u]:
            for nk, cost, disc in ((k, w, False), (k + 1, w // 2, True)):
                if nk <= D and d + cost < dist.get((v, nk), INF):
                    dist[(v, nk)] = d + cost; par[(v, nk)] = (u, k, disc, w); heapq.heappush(pq, (d + cost, v, nk))
    return -1, [], []

for args, want in examples:
    c, path, used = min_cost_with_path_ans(*args)
    assert c == want
    if c != -1:
        wmap = {}
        for a, b, w in args[1]: wmap[(a, b)] = wmap[(b, a)] = w
        tot = sum(wmap[(a, b)] for a, b in zip(path, path[1:])) - sum(w - w // 2 for _, _, w in used)
        assert tot == c and len(used) <= args[2]
    print(f"chi phí {c}, đường {path}, cạnh dùng phiếu {used}")
print("Bài 1-4: OK")

chi phí 9, đường [0, 1, 4], cạnh dùng phiếu [(1, 4, 11)]
chi phí 8, đường [0, 1, 2, 3], cạnh dùng phiếu [(0, 1, 6), (1, 2, 7), (2, 3, 5)]
chi phí -1, đường [], cạnh dùng phiếu []
Bài 1-4: OK


In [23]:
# Bài 5 — hai loại phiếu
def two_coupons(n, highways, A, B):
    g = build_graph(n, highways); A = min(A, n - 1); B = min(B, n - 1); INF = float("inf")
    dist = {(0, 0, 0): 0}; pq = [(0, 0, 0, 0)]
    while pq:
        d, u, a, b = heapq.heappop(pq)
        if d > dist[(u, a, b)]: continue
        if u == n - 1: return d
        for v, w in g[u]:
            for na, nb, cost in ((a, b, w), (a + 1, b, w // 2), (a, b + 1, 0)):
                if na <= A and nb <= B and d + cost < dist.get((v, na, nb), INF):
                    dist[(v, na, nb)] = d + cost; heapq.heappush(pq, (d + cost, v, na, nb))
    return -1

def two_coupons_brute(n, highways, A, B):
    g = build_graph(n, highways); best = float("inf"); seen = [False] * n
    def on_path(ws):
        ws = sorted(ws, reverse=True)
        free = ws[:B]; rest = ws[B:]
        return sum(w // 2 for w in rest[:A]) + sum(rest[A:])
    def dfs(u, ws):
        nonlocal best
        if u == n - 1: best = min(best, on_path(ws)); return
        for v, w in g[u]:
            if not seen[v]:
                seen[v] = True; ws.append(w); dfs(v, ws); ws.pop(); seen[v] = False
    seen[0] = True; dfs(0, [])
    return -1 if best == float("inf") else best

rng = random.Random(55)
for _ in range(300):
    n = rng.randint(2, 6); pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    hw = [[a, b, rng.randint(0, 30)] for a, b in rng.sample(pairs, rng.randint(1, len(pairs)))]
    A, B = rng.randint(0, 3), rng.randint(0, 2)
    assert two_coupons(n, hw, A, B) == two_coupons_brute(n, hw, A, B), (n, hw, A, B)
print("Bài 5: hai loại phiếu khớp brute force; ví dụ 1 với A = 1, B = 1:", two_coupons(*ex1, 1, 1))

Bài 5: hai loại phiếu khớp brute force; ví dụ 1 với A = 1, B = 1: 2


## 20. Template nhớ nhanh + Checklist

```python
# Dijkstra trên đồ thị phân lớp (đỉnh, số quyền đặc biệt đã dùng)
D = min(D, n - 1)
dist = [[INF] * (D + 1) for _ in range(n)]; dist[src][0] = 0
pq = [(0, src, 0)]
while pq:
    d, u, k = heappop(pq)
    if d > dist[u][k]: continue
    if u == dst: return d
    for v, w in g[u]:
        relax(v, k,     d + w)          # không dùng quyền
        if k < D:
            relax(v, k + 1, d + w // 2) # dùng quyền: lên lớp
return -1
```

**Checklist trước khi submit:**
- [ ] Trạng thái gồm **(thành phố, số phiếu đã dùng)**, không đánh dấu theo thành phố.
- [ ] Cạnh hai chiều.
- [ ] Giảm giá `w // 2` (làm tròn xuống).
- [ ] Đáp án là `min` theo mọi số phiếu (dừng ở lần đầu lấy ra đích).
- [ ] Cắt `D` về `n − 1`; trả `-1` khi không tới được.